# Predicting warranty claims within a year
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), 5-fold stratified CV on training rows only, selection by mean CV ROC AUC, holdout scored once.

`device_serial` is excluded (identifier). The remaining features (channel, device age, price, daily hours) are recorded at sale/registration, so they are known at prediction time.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

sha = lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert sha('data.csv')=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert sha('holdout.csv')=='083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==7000 and len(hold_idx)==1400 and len(set(hold_idx))==1400 and hold_idx.max()<len(df)
TARGET='claim_within_year'
FEATURES=['channel','device_age_years','price_eur','daily_hours']
is_hold = np.zeros(len(df),bool); is_hold[hold_idx]=True
train_idx = np.where(~is_hold)[0]
X, y = df[FEATURES], df[TARGET]
print(len(train_idx), 'training rows;', len(hold_idx), 'holdout rows; train positive rate', y.iloc[train_idx].mean().round(3))

## Training-data exploration (training rows only)

In [ ]:
tr = df.iloc[train_idx]
print(tr.groupby('channel')[TARGET].agg(['mean','size']))
print(tr[FEATURES[1:]+[TARGET]].corr()[TARGET])

## Candidates (named before any scoring)
All preprocessing is inside the pipeline, so it is refit in each CV fold.

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
def prep(scale):
    return ColumnTransformer([('num', Pipeline([('imp',SimpleImputer(strategy='median'))]+([('sc',StandardScaler())] if scale else [])), num),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
candidates = {
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=2000))]),
 'logreg_C1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=1,max_iter=2000))]),
 'random_forest': Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=1))]),
 'hist_gb': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

## Cross-validation on training rows

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
folds = [(train_idx[a], train_idx[b]) for a,b in skf.split(train_idx, y.iloc[train_idx])]
cv_scores = {}
for name, est in candidates.items():
    cv_scores[name] = [float(roc_auc_score(y.iloc[va], clone(est).fit(X.iloc[t], y.iloc[t]).predict_proba(X.iloc[va])[:,1])) for t,va in folds]
cv = pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1)
print(cv.sort_values('mean',ascending=False).round(4))

## Choose by mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
final = clone(candidates[chosen]).fit(X.iloc[train_idx], y.iloc[train_idx])
holdout_auc = float(roc_auc_score(y.iloc[hold_idx], final.predict_proba(X.iloc[hold_idx])[:,1]))
print('chosen:', chosen, '| mean CV AUC', round(np.mean(cv_scores[chosen]),4), '| HOLDOUT AUC', round(holdout_auc,4))

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[t.tolist(), va.tolist()] for t,va in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': hold_idx.tolist(),
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))